In [1]:
import os
import json
from pathlib import Path
from tau2_enhanced.analysis.analyzer import LogAnalyzer
from tau2_enhanced.analysis.visualizer import LogVisualizer
from tau2_enhanced.logging.events import ToolExecutionEvent

SCRIPT_DIR = Path('.').parent.resolve()
PROJECT_ROOT = SCRIPT_DIR.parent
LOG_FILE = PROJECT_ROOT / "samples/logs/baseline_airline_xai_grok3_gemini2_5_flash_reduced.json"

2025-10-22 10:15:29.773 | INFO     | tau2.utils.utils:<module>:27 - Using data directory from source: /Users/jitraychowdhury/Documents/code_personal/ai_agent_analysis/tau2/tau2-bench/data
2025-10-22 10:15:31.021 | INFO     | tau2.utils.llm_utils:<module>:65 - LiteLLM: Cache is disabled
2025-10-22 10:15:31.022 | WARNING  | tau2.utils.llm_utils:<module>:72 - Sonnet thinking is disabled
2025-10-22 10:15:31.067 | DEBUG    | tau2.registry:<module>:174 - Registering default components...
2025-10-22 10:15:31.068 | DEBUG    | tau2.registry:<module>:194 - Default components registered successfully. Registry info: {
  "domains": [
    "mock",
    "airline",
    "retail",
    "telecom",
    "telecom-workflow"
  ],
  "agents": [
    "llm_agent",
    "llm_agent_gt",
    "llm_agent_solo"
  ],
  "users": [
    "user_simulator",
    "dummy_user"
  ],
  "task_sets": [
    "mock",
    "airline",
    "retail",
    "telecom_full",
    "telecom_small",
    "telecom",
    "telecom-workflow"
  ]
}
2025-10-22

In [2]:

def analyze_logs(log_file: Path):
    """
    Loads, analyzes, and visualizes execution logs from a file.
    """
    print(f"📁 Loading logs from: {log_file}")

    try:
        with log_file.open('r') as f:
            data = json.load(f)
        print(f"  ✅ Successfully loaded log file")
    except (FileNotFoundError, json.JSONDecodeError) as e:
        print(f"❌ Error loading log file: {e}")
        return

    # Detect log format and handle accordingly
    if 'execution_events' in data:
        # New simplified format: direct execution_events array
        print("  📊 Detected new simplified enhanced logs format")

        # Try to load the corresponding standard results file for simulation success data
        if '_enhanced_logs' in log_file.name:
            standard_file = Path(str(log_file).replace('_enhanced_logs', ''))
            try:
                with standard_file.open('r') as f:
                    standard_data = json.load(f)
                print(f"  ✅ Loaded standard results file for simulation data")

                # Add simulation data and tasks to the enhanced logs for analysis
                data['simulations'] = standard_data.get('simulations', [])
                data['tasks'] = standard_data.get('tasks', [])

            except (FileNotFoundError, json.JSONDecodeError) as e:
                print(f"  ⚠️  Could not load standard results file: {e}")

        analyzer_data = data  # Use combined data
    elif 'simulations' in data:
        # Legacy format: simulations with embedded logs
        print("  📊 Detected legacy enhanced logs format")
        analyzer_data = data
    else:
        print("❌ Unrecognized log format - no execution_events or simulations found")
        return

    # The LogAnalyzer is capable of handling different log formats.
    # We pass the entire loaded JSON data to it.
    analyzer = LogAnalyzer(analyzer_data)
    return analyzer 
analyzer = analyze_logs(LOG_FILE)

📁 Loading logs from: /Users/jitraychowdhury/Documents/code_personal/ai_agent_analysis/tau2/tau2-enhanced/samples/logs/baseline_airline_xai_grok3_gemini2_5_flash_reduced.json
  ✅ Successfully loaded log file
  📊 Detected legacy enhanced logs format


In [3]:
analyzer.get_summary_metrics()

{'total_simulations': 200,
 'successful_simulations': 115,
 'task_success_rate': 0.575,
 'total_trials': 4,
 'total_tasks': 50,
 'total_tool_calls': 1162,
 'successful_calls': 384,
 'failed_calls': 204,
 'tool_success_rate': 0.6530612244897959,
 'tool_error_rate': 0.34693877551020413,
 'total_execution_time': np.float64(0.31214165687561035),
 'average_execution_time': np.float64(0.00026862448956592976),
 'median_execution_time': np.float64(5.3882598876953125e-05),
 'state_changing_calls': 139,
 'read_only_calls': 1023,
 'most_common_tool': 'get_reservation_details',
 'slowest_tool_avg': 'get_user_details',
 'fastest_tool_avg': 'transfer_to_human_agents',
 'execution_timespan': 7286.293608,
 'tools_used': 13,
 'success_metric_source': 'action_checks'}

In [4]:
analyzer.get_tool_performance()

Tool Performance: source of success rate -  action_checks


,tool_name,total_calls,successful_calls,avg_execution_time,median_execution_time,min_execution_time,max_execution_time,total_execution_time,state_changing_calls,avg_result_size,failed_calls,success_rate,error_rate,state_change_rate,performance_category
4,get_reservation_details,488,217,0.000101,0.000049,0.000027,0.023534,0.049372,0,200.000000,11.0,0.444672,0.022541,0.0,poor
6,search_direct_flight,164,17,0.000236,0.000225,0.000169,0.000932,0.038724,0,88.926829,63.0,0.103659,0.384146,0.0,poor
5,get_user_details,158,56,0.000823,0.000050,0.000037,0.067323,0.130068,0,200.000000,0.0,0.354430,0.000000,0.0,poor
7,search_onestop_flight,100,100,0.000681,0.000703,0.000170,0.002618,0.068130,0,120.800000,0.0,1.000000,0.000000,0.0,excellent
11,update_reservation_flights,62,38,0.000132,0.000120,0.000093,0.000755,0.008206,62,158.064516,46.0,0.612903,0.741935,1.0,poor
3,get_flight_status,56,56,0.000062,0.000054,0.000046,0.000407,0.003467,0,7.714286,0.0,1.000000,0.000000,0.0,excellent
9,transfer_to_human_agents,56,0,0.000060,0.000047,0.000038,0.000759,0.003381,0,19.000000,4.0,0.000000,0.071429,0.0,poor
2,cancel_reservation,39,29,0.000148,0.000129,0.000105,0.000533,0.005785,39,200.000000,22.0,0.743590,0.564103,1.0,poor
10,update_reservation_baggages,16,9,0.000079,0.000073,0.000060,0.000132,0.001264,16,200.000000,15.0,0.562500,0.937500,1.0,poor
0,book_reservation,9,5,0.000216,0.000215,0.000197,0.000249,0.001948,9,177.777778,28.0,0.555556,3.111111,1.0,poor


In [5]:
analyzer.get_tool_sequence_analysis()

,source,target,count
0,get_reservation_details,get_reservation_details,287
1,get_user_details,get_reservation_details,134
2,search_direct_flight,search_onestop_flight,68
3,search_direct_flight,search_direct_flight,66
4,search_onestop_flight,search_direct_flight,45
...,...,...,...
62,get_flight_status,search_direct_flight,1
61,cancel_reservation,update_reservation_flights,1
60,update_reservation_flights,transfer_to_human_agents,1
59,update_reservation_baggages,cancel_reservation,1


In [6]:
analyzer.get_tool_usage_patterns()

{'total_unique_tools': 13,
 'most_used_tool': {'name': 'get_reservation_details',
  'calls': np.int64(488),
  'percentage': np.float64(41.996557659208264)},
 'least_used_tool': {'name': 'calculate',
  'calls': np.int64(1),
  'percentage': np.float64(0.08605851979345956)},
 'tool_usage_distribution': {'get_reservation_details': 488,
  'search_direct_flight': 164,
  'get_user_details': 158,
  'search_onestop_flight': 100,
  'update_reservation_flights': 62,
  'transfer_to_human_agents': 56,
  'get_flight_status': 56,
  'cancel_reservation': 39,
  'update_reservation_baggages': 16,
  'book_reservation': 9,
  'update_reservation_passengers': 9,
  'send_certificate': 4,
  'calculate': 1},
 'diversity_index': np.float64(1.845577532194016),
 'usage_concentration': 'distributed'}

In [7]:
analyzer.get_error_pattern_analysis()

{'total_errors': 14,
 'error_rate': 0.012048192771084338,
 'tools_with_errors': 2,
 'error_types': {},
 'most_common_error': None,
 'avg_error_execution_time': np.float64(0.00011995860508510045),
 'avg_success_execution_time': np.float64(0.00027043748815715933),
 'error_time_correlation': 'shorter',
 'tools_by_error_rate': {'update_reservation_flights': 0.20967741935483872,
  'book_reservation': 0.1111111111111111,
  'calculate': nan,
  'cancel_reservation': nan,
  'get_flight_status': nan,
  'get_reservation_details': nan,
  'get_user_details': nan,
  'search_direct_flight': nan,
  'search_onestop_flight': nan,
  'send_certificate': nan,
  'transfer_to_human_agents': nan,
  'update_reservation_baggages': nan,
  'update_reservation_passengers': nan}}

In [8]:
analyzer.get_requestor_analysis()

{'requestor_breakdown': {'assistant': {'total_calls': 1162,
   'success_rate': np.float64(0.9879518072289156),
   'avg_execution_time': np.float64(0.00026862448956592976),
   'state_change_rate': np.float64(0.11962134251290878),
   'most_used_tools': {'get_reservation_details': 488,
    'search_direct_flight': 164,
    'get_user_details': 158},
   'error_rate': np.float64(0.012048192771084338)}},
 'total_requestors': 1}

In [9]:
analyzer.get_advanced_statistics()

{'execution_time_stats': {'mean': np.float64(0.00026862448956592976),
  'median': np.float64(5.3882598876953125e-05),
  'std': np.float64(0.002463581980206369),
  'min': np.float64(2.7418136596679688e-05),
  'max': np.float64(0.0673227310180664),
  'p25': np.float64(4.8160552978515625e-05),
  'p75': np.float64(0.00017017126083374023),
  'p95': np.float64(0.00070185661315918),
  'p99': np.float64(0.0013212800025939944)},
 'success_rate_confidence': {'rate': np.float64(0.9879518072289156),
  'lower_bound': np.float64(0.9816787125671287),
  'upper_bound': np.float64(0.9942249018907026),
  'sample_size': 1162},
 'tool_distribution': {'entropy': np.float64(1.845577532194016),
  'gini_coefficient': np.float64(0.6222692969680922),
  'concentration_ratio': np.float64(0.4199655765920826)}}